In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
data_1 = pd.read_csv("./data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
data_2 = pd.read_csv("./data/2023-02-12.csv")

## Préparation des données:

In [ ]:

data_1.columns = data_1.columns.str.replace(" ", "").str.lower()
data_2.columns = data_2.columns.str.replace(" ", "").str.lower()

common_columns = list(set(data_1.columns) & set(data_2.columns))

data_1 = data_1[common_columns]
data_2 = data_2[common_columns]

concatenated_data = pd.concat([data_1, data_2], ignore_index=True)

# Ajout de l'index temporel simulé, il va nous servir pour l'entrainement du CNN.
concatenated_data['time_index'] = concatenated_data.index


## Phase 1 : Détection d'Attaques avec Random Forest

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Sélection des caractéristiques pour Random Forest
features = [col for col in concatenated_data.columns if col != 'label' and col != 'time_index']
X = concatenated_data[features]
y = concatenated_data['label']

# Normalisation des caractéristiques
scaler = StandardScaler()
# Vérifier la présence de NaN ou de valeurs infinies
print("Valeurs NaN avant nettoyage :", X.isna().sum().sum())
print("Valeurs infinies avant nettoyage :", np.isinf(X).sum().sum())

# Remplacer les valeurs infinies par des grandes valeurs finies
X.replace([np.inf, -np.inf], np.nan, inplace=True)

# Remplacer les NaN par la moyenne de chaque colonne
X.fillna(X.mean(), inplace=True)

# Vérifier après le nettoyage
print("Valeurs NaN après nettoyage :", X.isna().sum().sum())
print("Valeurs infinies après nettoyage :", np.isinf(X).sum().sum())
X_scaled = scaler.fit_transform(X)

# Division des données en jeu d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Entraînement du modèle Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Prédictions et évaluation
y_pred_rf = rf_model.predict(X_test)
print("Performance de Random Forest :")
print(classification_report(y_test, y_pred_rf))


Valeurs NaN avant nettoyage : 85
Valeurs infinies avant nettoyage : 247


C:\Users\ibrah\AppData\Local\Temp\ipykernel_45512\2175874002.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.replace([np.inf, -np.inf], np.nan, inplace=True)
C:\Users\ibrah\AppData\Local\Temp\ipykernel_45512\2175874002.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.fillna(X.mean(), inplace=True)


Valeurs NaN après nettoyage : 0
Valeurs infinies après nettoyage : 0
Performance de Random Forest :
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     19448
        DDoS       1.00      1.00      1.00     25739
    adbhoney       0.94      0.70      0.81        44
    ciscoasa       0.77      0.59      0.67        41
      cowrie       0.95      0.96      0.96       428
     ddospot       1.00      1.00      1.00     14380
  elasticpot       0.90      0.45      0.60        20
     log4pot       0.86      0.96      0.91       291
    mailoney       0.86      0.55      0.67        11
    redispot       1.00      0.38      0.55         8

    accuracy                           1.00     60410
   macro avg       0.93      0.76      0.81     60410
weighted avg       1.00      1.00      1.00     60410



## Phase 2: Préparation des données pour le CNN

In [17]:
sequence_length = 20  # Taille des séquences pour CNN
features = [col for col in concatenated_data.columns if col != 'label' and col != 'time_index']

sequences = []
labels = []

for i in range(len(concatenated_data) - sequence_length):
    seq = concatenated_data.iloc[i:i+sequence_length][features].values  # Extraire une séquence
    label = concatenated_data.iloc[i+sequence_length]['label']  # Dernière valeur comme cible
    
    sequences.append(seq)
    labels.append(label)

# Conversion en tableau numpy
sequences = np.array(sequences)
labels = np.array(labels)

# Division des données en entraînement et test
X_train_cnn, X_test_cnn, y_train_cnn, y_test_cnn = train_test_split(sequences, labels, test_size=0.2, random_state=42)

print(f"Taille des données CNN : {X_train_cnn.shape}, {y_train_cnn.shape}")


Taille des données CNN : (241622, 20, 57), (241622,)


### Entrainement du CNN

In [20]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
import numpy as np

# 🔹 Étape 1 : Encodage des labels en entiers
label_encoder = LabelEncoder()
y_train_cnn = label_encoder.fit_transform(y_train_cnn)  # Convertit les classes texte en entiers
y_test_cnn = label_encoder.transform(y_test_cnn)  # Transforme les labels de test selon le même encodage

# Vérification des classes encodées
print("Classes encodées :", list(label_encoder.classes_))  

# 🔹 Étape 2 : Encodage en One-Hot pour la classification multi-classes
num_classes = len(label_encoder.classes_)  # Nombre total de classes uniques
y_train_cnn = to_categorical(y_train_cnn, num_classes)
y_test_cnn = to_categorical(y_test_cnn, num_classes)

# 🔹 Étape 3 : Construction du modèle CNN 1D
model = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(sequence_length, len(features))),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=32, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')  # Classification multi-classes
])

# Étape 4 : Compilation du modèle
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Étape 5 : Entraînement du modèle
history = model.fit(X_train_cnn, y_train_cnn, epochs=10, batch_size=32, validation_data=(X_test_cnn, y_test_cnn))

# Étape 6 : Évaluation du modèle
loss, accuracy = model.evaluate(X_test_cnn, y_test_cnn)
print(f"Performance du CNN 1D - Accuracy: {accuracy:.4f}")


Classes encodées : [np.str_('BENIGN'), np.str_('DDoS'), np.str_('adbhoney'), np.str_('ciscoasa'), np.str_('cowrie'), np.str_('ddospot'), np.str_('elasticpot'), np.str_('log4pot'), np.str_('mailoney'), np.str_('redispot')]


c:\Users\ibrah\Desktop\IDIA-5A\PRED\PRED\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
7551/7551 ━━━━━━━━━━━━━━━━━━━━ 37s 5ms/step - accuracy: 0.3241 - loss: nan - val_accuracy: 0.3240 - val_loss: nan
Epoch 2/10
7551/7551 ━━━━━━━━━━━━━━━━━━━━ 34s 5ms/step - accuracy: 0.3257 - loss: nan - val_accuracy: 0.3240 - val_loss: nan
Epoch 3/10
7551/7551 ━━━━━━━━━━━━━━━━━━━━ 35s 5ms/step - accuracy: 0.3238 - loss: nan - val_accuracy: 0.3240 - val_loss: nan
Epoch 4/10
7551/7551 ━━━━━━━━━━━━━━━━━━━━ 35s 5ms/step - accuracy: 0.3238 - loss: nan - val_accuracy: 0.3240 - val_loss: nan
Epoch 5/10
7551/7551 ━━━━━━━━━━━━━━━━━━━━ 37s 5ms/step - accuracy: 0.3238 - loss: nan - val_accuracy: 0.3240 - val_loss: nan
Epoch 6/10
7551/7551 ━━━━━━━━━━━━━━━━━━━━ 38s 5ms/step - accuracy: 0.3231 - loss: nan - val_accuracy: 0.3240 - val_loss: nan
Epoch 7/10
7551/7551 ━━━━━━━━━━━━━━━━━━━━ 37s 5ms/step - accuracy: 0.3212 - loss: nan - val_accuracy: 0.3240 - val_loss: nan
Epoch 8/10
7551/7551 ━━━━━━━━━━━━━━━━━━━━ 35s 5ms/step - accuracy: 0.3230 - loss: nan - val_accuracy: 0.3240 - val_loss: nan


## Fusion des deux modèle (Random Forest && CNN)

In [ ]:
# Prédictions de Random Forest
y_pred_rf_test = rf_model.predict(X_test)
rf_probabilities = rf_model.predict_proba(X_test)  # Probabilités des classes

# Prédictions du CNN 1D
cnn_probabilities = model.predict(X_test_cnn)

# Fusion des probabilités (moyenne des deux modèles)
combined_probabilities = (rf_probabilities + cnn_probabilities) / 2
y_pred_final = np.argmax(combined_probabilities, axis=1)

# Évaluation finale du modèle hybride
print("Performance du modèle combiné Random Forest + CNN 1D :")
print(classification_report(np.argmax(y_test_cnn, axis=1), y_pred_final))
